![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E03 - LCEL Chain: componer con `|` (Resolution)

## BLOQUE 1 — ¿Qué es LCEL?

LCEL significa **LangChain Expression Language** y es la forma de conectar componentes de LangChain usando el operador `|` (pipe).

### Sintaxis

```python
chain = prompt | llm | parser
```

El flujo de datos va de izquierda a derecha:

```text
Input (dict)  ->  PromptTemplate  ->  ChatOpenAI  ->  StrOutputParser  ->  Output (str)
                    formatea          genera            extrae
                    variables         respuesta         el texto
```

### ¿Por qué LCEL y no funciones imperativas?

```text
SIN LCEL (imperativo):                         CON LCEL (declarativo):
                                              
messages = prompt.format(question=q)           chain = prompt | llm | parser
ai_msg = llm.invoke(messages)                 
text = ai_msg.content                         result = chain.invoke({"question": q})
result = parser.invoke(text)                  
"""
4 lineas para entender el flujo.              1 linea para entender el flujo.
El modelo esta hardcodeado dentro de la       El modelo es un componente intercambiable.
funcion.
"""
```

### Beneficios automáticos que da LCEL

| Capacidad | Código | Sin LCEL tendrías que... |
|---|---|---|
| **Streaming** | `chain.stream(input)` | Implementar un callback manual |
| **Batch** | `chain.batch([q1, q2, q3])` | Hacer un loop for |
| **Trazabilidad** | `chain.invoke(input)` | Agregar prints en cada paso |
| **Paralelismo** | `RunnableParallel` | Usar threading manual |
| **Fallback** | `chain.with_fallbacks(...)` | Escribir try/except en cada paso |

> **En resumen**: LCEL no es solo "syntactic sugar" — LCEL te da capacidades de producción automáticamente.

## BLOQUE 2 — Setup: API key

In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")

## BLOQUE 3 — Sin LangChain: el script imperativo

Primero veamos el enfoque táctico que traemos de M3L1 para entender todo lo que LCEL resuelve.

In [ ]:
from openai import OpenAI

client = OpenAI()

def answer_without_langchain(question: str) -> str:
    # Paso 1: construir el prompt manualmente
    system_msg = "Eres un asistente util. Responde de forma concisa."
    user_msg = question

    # Paso 2: estructurar mensajes en formato específico de OpenAI
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    # Paso 3: llamar a la API con modelo hardcodeado
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,
    )

    # Paso 4: extraer texto de una respuesta anidada
    answer = response.choices[0].message.content
    return answer


respuesta = answer_without_langchain("Cual es la capital de Francia?")
print(f"Respuesta: {respuesta}")

### Los 4 problemas de este enfoque

1. **Modelo hardcodeado**: `model="gpt-4o-mini"` — si querés cambiar a `gpt-4o`, buscás en todas las funciones
2. **Formato específico de OpenAI**: si después usás Anthropic, el formato de mensajes cambia
3. **Extracción manual**: `response.choices[0].message.content` — propenso a errores si la API cambia
4. **No componible**: no podés conectar esto con un retriever, memoria, o parser nuevo sin reescribir

Esto es lo que la lecture llama **"scripting táctico"**: funciona, pero no escala.

## BLOQUE 4 — Componentes individuales de LangChain

Antes de armar la chain, creamos cada componente por separado para entender qué hace cada uno.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. LLM WRAPPER: encapsula el modelo
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. PARSER: extrae el texto de la respuesta del modelo
parser = StrOutputParser()

print(f"LLM configurado: {llm.model_name}")
print(f"Parser: {type(parser).__name__}")

### Componente 1: ChatOpenAI (el modelo)

`ChatOpenAI` es un **wrapper** que encapsula el modelo. Podés pasarle:
- `model`: nombre del modelo (`"gpt-4o-mini"`, `"gpt-4o"`, etc.)
- `temperature`: control de creatividad (0 = determinista, 1 = muy creativo)
- `api_key`: opcional, si no está en `OPENAI_API_KEY`

La interfaz estándar es `.invoke(input)` que devuelve un `AIMessage`.

### Componente 2: StrOutputParser (el extractor)

Toma el `AIMessage` y devuelve solo `.content` como string. Sin esto, tendrías que escribir `.content` en cada lugar.

### Componente 3: ChatPromptTemplate (el formateador)

Arma los mensajes con roles (system, human) y variables explícitas `{question}`.

## BLOQUE 5 — TODO 1: Crear el PromptTemplate

In [ ]:
# TODO 1: crear el ChatPromptTemplate con variable {question}
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente util. Responde de forma concisa."),
    ("human", "{question}"),
])

print(f"Prompt creado: {type(prompt).__name__}")
print(f"Variables: {prompt.input_variables}")
print()
print("--- Debug: formato del prompt ---")
msgs = prompt.format_messages(question="pregunta de prueba")
for m in msgs:
    print(f"  [{m.type}] {m.content}")

## BLOQUE 6 — TODO 2: Componer la chain con LCEL

El operador `|` pasa el output de un componente al input del siguiente:

```text
prompt | llm | parser
  ^       ^      ^
  |       |      +-- recibe AIMessage, devuelve str
  |       +--------- recibe lista de mensajes, devuelve AIMessage
  +----------------- recibe dict, devuelve lista de mensajes
```

**Tres componentes, dos pipes, una chain.**

In [ ]:
# TODO 2: componer la chain con LCEL
chain = prompt | llm | parser

print(f"Tipo de la chain: {type(chain).__name__}")
print()
print("La chain completa acepta .invoke(), .stream(), .batch()")

## BLOQUE 7 — TODO 3: Invocar la chain

`.invoke()` en una chain recibe un dict con las variables del prompt y devuelve el string ya parseado.

In [ ]:
# TODO 3: invocar la chain
respuesta = chain.invoke({"question": "Cual es la capital de Francia?"})
print(f"Respuesta: {respuesta}")
print(f"Tipo de respuesta: {type(respuesta)}")
print()
print("StrOutputParser garantiza que la respuesta sea string.")

## BLOQUE 8 — El modelo es reemplazable

Esta es una de las ventajas clave de LangChain: **el prompt y el parser no saben qué modelo los conecta**. Solo necesitan que el componente del medio tenga `.invoke()`.

Podés cambiar el modelo, la temperatura, o incluso el proveedor, sin tocar el resto de la chain.

In [ ]:
# Mismo prompt, mismo parser, LLM diferente
llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
chain_v2 = prompt | llm_v2 | parser

respuesta_v2 = chain_v2.invoke({"question": "Cual es la capital de Francia?"})
print(f"Con temperatura 0.5: {respuesta_v2}")
print()
print("Solo cambiamos el componente LLM.")
print("El prompt y el parser son los mismos objetos.")

## BLOQUE 9 — Capacidades extra que LCEL da gratis

Al usar `|`, LangChain envuelve la chain en un `RunnableSequence` que expone automáticamente:

In [ ]:
print("========== CAPACIDADES DE LA CHAIN ==========")
print(f"1. .invoke():         {chain.invoke({'question': 'Di hola'})}")
print()

# 2. Streaming: muestra tokens de a uno
print("2. .stream():         ", end="")
for chunk in chain.stream({"question": "Cuenta hasta 5 en numeros"}):
    print(chunk, end="", flush=True)
print("\n")

# 3. Batch: múltiples inputs en paralelo
print("3. .batch():          ")
preguntas = [
    {"question": "Capital de Francia?"},
    {"question": "Capital de Argentina?"},
    {"question": "Capital de Japon?"},
]
respuestas = chain.batch(preguntas)
for p, r in zip(preguntas, respuestas):
    print(f"     {p['question']} -> {r}")
print()

print("4. .get_graph():      ", chain.get_graph().print_ascii())
print()
print("Todo esto SIN escribir una sola linea extra de codigo.")

## BLOQUE 10 — Comparación final

| Aspecto | Sin LangChain (imperativo) | Con LangChain (LCEL) |
|---|---|---|
| **Construcción del prompt** | f-string o concatenación | `ChatPromptTemplate` con variables explícitas |
| **Llamada al modelo** | `client.chat.completions.create(...)` | `llm.invoke(...)` |
| **Extraer texto** | `response.choices[0].message.content` | `StrOutputParser` automático |
| **Flujo de datos** | Implícito (hay que leer 4 líneas) | Explícito (`prompt \| llm \| parser`) |
| **Cambiar modelo** | Buscar y reemplazar en cada función | Cambiar el objeto `llm` |
| **Streaming** | Implementar manual | `chain.stream()` gratis |
| **Batch** | Loop for manual | `chain.batch()` gratis |
| **Testing** | Mockear toda la API | Usar `FakeListChatModel` |

### ¿Cuándo usar LCEL?

- Siempre que tengas **2+ componentes** que conectar
- Cuando necesites **streaming o batch**
- Cuando el modelo pueda **cambiar en el futuro**
- Cuando quieras **testear** componentes por separado

### ¿Cuándo NO?

- Prototipos de 1 celda en una notebook (aunque igual conviene por claridad)

## BLOQUE 11 — Checks automáticos

In [ ]:
def run_checks():
    assert prompt is not None
    assert chain is not None
    test_response = chain.invoke({"question": "Di solo la palabra 'test'"})
    assert isinstance(test_response, str)
    assert len(test_response) > 0
    r1 = chain.invoke({"question": "Cual es 2 + 2? Responde solo el numero"})
    assert "4" in r1
    print("M3L2 E03 Resolution checks passed")

run_checks()

## Cierre

Hoy aprendiste:

1. **LCEL** = LangChain Expression Language. Usa `|` para conectar componentes
2. **`prompt | llm | parser`** es la chain más simple: formatea -> genera -> extrae
3. **Streaming, batch y tracing** vienen gratis con LCEL
4. **El modelo es reemplazable** sin tocar prompt ni parser
5. **El flujo de datos es explícito**: se lee en una línea

### Próximos pasos

- **E04**: agregar memoria a la chain con `RunnableWithMessageHistory`
- **E10**: RAG completo donde la chain incluye un retriever
- **E05**: agregar tools a la chain con `bind_tools()`